# Baseline retrieval vs cross-encoder re-ranking

Document: FIA 2026 Formula 1 Regulations, Section B (Sporting), Issue 09, dated 1 October 2026.

Both pipelines run over the same chunked text and use the same prompt and the same language model.

1. Pipeline 1: embed the question with a bi-encoder, take the 10 nearest chunks by cosine
   similarity, send the top 3 to the model (Answer 1).
2. Pipeline 2: score the same 10 chunks with a cross-encoder, sort by that score, send the new
   top 3 to the model (Answer 2).

Changes made to the source before indexing (also listed in the README):

* Pages 1 to 3 (cover, colour legend, contents) are not indexed.
* Pages 95 to 101 (approved changes for 2027 to 2029, not in force) are not indexed.
* Struck-out text is removed. The PDF marks deleted rules with a magenta line through the text,
  and plain text extraction would index them as if they were still in force.

To run: install `requirements.txt`, then run all cells. The answer step needs `OPENCODE_API_KEY`
in the environment. Without it the notebook uses the saved answers in `results/answers_cache.json`.

## 1. Setup

In [ ]:
import hashlib, json, os, re, statistics, sys, time, urllib.request, uuid
from datetime import date
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pymupdf==1.28.2", "sentence-transformers==6.1.0", "faiss-cpu==1.15.1"], check=True)

# torch and faiss each bring their own OpenMP runtime. Import torch first: the other order
# crashes on Apple Silicon.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
import pymupdf

In [ ]:
PDF_URL = ("https://www.fia.com/system/files/documents/"
           "fia_2026_f1_regulations_-_section_b_sporting_-_iss_09_-_2026-10-01.pdf")
PDF_SHA256 = "5e156f50f1ca529d43551f52e82f5f2872bdfbe088e96fab39b861f6b0704b85"
PDF_PATH = Path("data/fia_2026_f1_section_b_sporting_iss09.pdf")
FIRST_PAGE, LAST_PAGE = 4, 94          # 1-based, inclusive

BI_ENCODER = "BAAI/bge-small-en-v1.5"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"
# bge asks for this phrase in front of short queries. Passages get no phrase, and the
# cross-encoder reads the plain question.
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
TOP_K, TOP_N = 10, 3
MAX_CHUNK_TOKENS = 400                 # leaves room for the question in the cross-encoder's 512

GEN_MODEL = "claude-haiku-5-5"
GEN_URL = "https://opencode.ai/inference/go/anthropic/v1/messages"
CACHE_PATH = Path("results/answers_cache.json")
CACHE_URL = ("https://raw.githubusercontent.com/abhinavborah/rag-rerank-eval/main/"
             "results/answers_cache.json")      # used when the notebook runs without the repository

## 2. Source document

In [ ]:
def sha256_of(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

PDF_PATH.parent.mkdir(exist_ok=True)
if not PDF_PATH.exists():
    req = urllib.request.Request(PDF_URL, headers={"User-Agent": "Mozilla/5.0"})
    PDF_PATH.write_bytes(urllib.request.urlopen(req, timeout=120).read())
assert sha256_of(PDF_PATH) == PDF_SHA256, "the downloaded PDF is not the pinned edition"
doc = pymupdf.open(PDF_PATH)
print(doc.metadata["title"], "|", len(doc), "pages")

## 3. Text extraction

Each page is read character by character. A character is dropped when a magenta bar crosses the
middle of it, which is how the PDF shows deleted text. Headers and footers are skipped. What is
left is a list of visual lines with their position and font, which the next step groups into
clauses.

In [ ]:
MAGENTA = (1.0, 0.0, 1.0)
BLUE = 0x2D5F                          # colour of clause labels and headings
LABEL_COLORS = (BLUE, 0xFF00FF)        # a clause added in this issue has a magenta label

def strike_bars(page):
    bars = []
    for d in page.get_drawings():
        r, col = d["rect"], d.get("color") or d.get("fill")
        if r.height < 1.6 and r.width > 3 and col and tuple(round(c, 2) for c in col) == MAGENTA:
            bars.append(r)
    return bars

def is_struck(bbox, bars):
    x0, y0, x1, y1 = bbox
    cx = (x0 + x1) / 2
    lo, hi = y0 + (y1 - y0) * 0.25, y0 + (y1 - y0) * 0.75
    return any(lo <= (r.y0 + r.y1) / 2 <= hi and r.x0 - 1 <= cx <= r.x1 + 1 for r in bars)

def page_lines(page):
    """Visible lines of one page: dicts with y, x, size, bold, color and text."""
    bars = strike_bars(page)
    out, dropped = [], 0
    for block in page.get_text("rawdict")["blocks"]:
        for line in block.get("lines", []):
            y0 = line["bbox"][1]
            if y0 < 50 or y0 > page.rect.height - 65:       # running header and footer
                continue
            chars, first = [], None
            for span in line["spans"]:
                for ch in span["chars"]:
                    if is_struck(ch["bbox"], bars):
                        dropped += 1
                        continue
                    chars.append(ch["c"])
                    if first is None and ch["c"].strip():
                        first = (span, ch["bbox"][0])
            text = re.sub(r"\s+", " ", "".join(chars)).strip()
            if not text or first is None:
                continue
            span, x = first
            if x < 85 and re.fullmatch(r"\d+(\.\d+)*", text):   # row numbers in the appendix tables
                continue
            out.append({"y": y0, "x": x, "size": span["size"], "bold": "Bold" in span["font"],
                        "color": span["color"], "text": text})
    return sorted(out, key=lambda l: (round(l["y"]), l["x"])), dropped

lines, struck_chars = [], 0
for pno in range(FIRST_PAGE - 1, LAST_PAGE):
    pl, n = page_lines(doc[pno])
    struck_chars += n
    for l in pl:
        l["page"] = pno + 1
    lines += pl
print(len(lines), "lines kept,", struck_chars, "struck characters removed")

full = " ".join(l["text"] for l in lines)
for marker in ("SECTION B: SPORTING REGULATIONS", "Fédération Internationale", "Issue 09",
               "2026 Formula 1 Regulations - Section B", "01 October 2026"):
    assert marker not in full, marker

## 4. Chunking

The regulations are numbered (B1.9.3 and so on). A chunk is one clause, with its article,
section and clause number written in front, so a chunk that says "the penalty" still carries its
heading. A clause longer than the token limit is split at paragraph boundaries, with the last
paragraph repeated at the start of the next piece. Paragraphs that are still too long are split
at sentence ends.

In [ ]:
tok = SentenceTransformer(BI_ENCODER, device="cpu").tokenizer     # token counts use the model's own tokenizer
tok.model_max_length = 10**6                                  # only counting tokens here, nothing is truncated

def n_tokens(text):
    return len(tok(text, add_special_tokens=True)["input_ids"])

def label_kind(l):
    t = l["text"]
    if l["bold"] and l["color"] in LABEL_COLORS and l["x"] < 60:
        if re.fullmatch(r"B\d+(\.\d+){2,}", t):
            return "clause"
        if re.fullmatch(r"B\d+\.\d+", t):
            return "section"
    if re.match(r"(ARTICLE|APPENDIX) B\d+:", t) and l["x"] < 60:
        return "article"
    return None

blocks = []          # one per clause (or per run of text under a heading)
state = {"article": None, "section": None, "sec_title": None, "clause": None}

def new_block():
    blocks.append({**state, "lines": []})

new_block()
last_section_y = None
for l in lines:
    kind = label_kind(l)
    if kind == "article":
        state.update(article=l["text"], section=None, sec_title=None, clause=None)
        new_block()
    elif kind == "section":
        state.update(section=l["text"], sec_title="", clause=None)
        last_section_y = (l["page"], round(l["y"]))
        new_block()
    elif kind == "clause":
        state.update(clause=l["text"])
        new_block()
    elif (last_section_y == (l["page"], round(l["y"])) and l["bold"] and l["color"] in LABEL_COLORS):
        state["sec_title"] = l["text"]
        blocks[-1]["sec_title"] = l["text"]
    else:
        blocks[-1]["lines"].append(l)

# Every line that looks like a clause or section number in the margin must have been read as a label,
# and every section must have a title.
missed = [l for l in lines if l["x"] < 60 and re.fullmatch(r"B\d+(\.\d+)+", l["text"]) and not label_kind(l)]
assert not missed, missed[:3]
assert all(b["sec_title"] for b in blocks if b["section"]), "a section lost its title"

SUBITEM = re.compile(r"^([a-z]|[ivx]+|\d+)\.\s")

def paragraphs(block_lines):
    paras, prev = [], None
    for l in block_lines:
        starts_new = (prev is None or SUBITEM.match(l["text"])
                      or (l["page"] == prev["page"] and l["y"] - prev["y"] > 18))
        if starts_new:
            paras.append([l["page"], [l["text"]]])
        else:
            paras[-1][1].append(l["text"])
        prev = l
    return [(" ".join(t), pg) for pg, t in paras]

def crumb(b):
    parts = [b["article"]]
    if b["section"]:
        parts.append(f"{b['section']} {b['sec_title'] or ''}".strip())
    if b["clause"]:
        parts.append(b["clause"])
    return " > ".join(p for p in parts if p)

def split_long(para, budget):
    sentences, pieces, cur = re.split(r"(?<=[.;:])\s+", para), [], ""
    for s in sentences:
        cand = f"{cur} {s}".strip()
        if cur and n_tokens(cand) > budget:
            pieces.append(cur)
            cur = s
        else:
            cur = cand
    return pieces + [cur]

# A heading block that holds only a short note (for example the committee line under an article
# title) is merged into the block after it, so it does not become a chunk of its own.
for i, b in enumerate(blocks[:-1]):
    if b["clause"] is None and b["lines"] and             n_tokens(" ".join(l["text"] for l in b["lines"])) < 45:
        blocks[i + 1]["lines"] = b["lines"] + blocks[i + 1]["lines"]
        b["lines"] = []

chunks, seen = [], {}
for b in blocks:
    paras = paragraphs(b["lines"])
    if not paras:
        continue
    head = crumb(b)
    title = paras[0][0] if len(paras[0][0].split()) <= 12 and not paras[0][0].endswith(".") else ""
    budget = MAX_CHUNK_TOKENS - n_tokens(head + " > " + title) - 2
    units = []                                   # (text, page) pairs
    for p, pg in paras:
        units += [(x, pg) for x in split_long(p, budget)] if n_tokens(p) > budget else [(p, pg)]
    groups, cur = [], []
    for u in units:
        if cur and n_tokens(" ".join(t for t, _ in cur + [u])) > budget:
            groups.append(cur)
            cur = [cur[-1], u] if n_tokens(cur[-1][0] + " " + u[0]) <= budget else [u]
        else:
            cur.append(u)
    groups.append(cur)
    base = b["clause"] or (b["section"] or (b["article"] or "front")).split(":")[0]
    seen[base] = seen.get(base, 0) + 1
    for k, g in enumerate(groups, 1):
        cid = base if len(groups) == 1 else f"{base}#{k}"
        if seen[base] > 1:
            cid += f"@{seen[base]}"
        part_head = head + " > " + title if (k > 1 and title) else head   # later pieces keep the clause title
        text = part_head + "\n" + "\n".join(t for t, _ in g)
        pages = [pg for _, pg in g]
        chunks.append({"id": cid, "text": text, "tokens": n_tokens(text), "pages": (min(pages), max(pages))})

sizes = [c["tokens"] for c in chunks]
print(len(chunks), "chunks | tokens min/median/max:", min(sizes), int(statistics.median(sizes)), max(sizes))
assert max(sizes) <= MAX_CHUNK_TOKENS + 10
assert len({c["id"] for c in chunks}) == len(chunks)

# The saved answers depend on the exact chunk text. If extraction or chunking changes, this fails.
corpus_hash = hashlib.sha256("\n".join(f"{c['id']}\t{c['text']}" for c in chunks).encode()).hexdigest()
EXPECTED_CHUNKS, EXPECTED_CORPUS_SHA256 = 375, "c7235f717f01eacef4ce35ff6b5d8ffe5ffbf851aab7b17824eea48ccd105b8e"
print("corpus hash", corpus_hash[:16])
if EXPECTED_CHUNKS is not None:
    assert (len(chunks), corpus_hash) == (EXPECTED_CHUNKS, EXPECTED_CORPUS_SHA256), "the chunked corpus changed"

## 5. Index and the two pipelines

In [ ]:
bi = SentenceTransformer(BI_ENCODER, device="cpu")
cross = CrossEncoder(CROSS_ENCODER, device="cpu", max_length=512)
assert max(c["tokens"] for c in chunks) <= bi.max_seq_length

emb = bi.encode([c["text"] for c in chunks], batch_size=32, normalize_embeddings=True,
                show_progress_bar=False).astype("float32")
index = faiss.IndexFlatIP(emb.shape[1])        # exact search; inner product of unit vectors = cosine
index.add(emb)
print(index.ntotal, "vectors of dimension", emb.shape[1])

# The cross-encoder cuts its input at 512 tokens. Check the longest chunk with a long question.
probe = " ".join(["word"] * 40)
longest = max(len(cross.tokenizer(probe, c["text"])["input_ids"]) for c in chunks)
assert longest <= 512, longest
print("longest question + chunk pair:", longest, "tokens")

# FAISS must agree with a plain matrix product, since the search is exact.
for q in ("pit lane speed limit", "tyre allocation for the race"):
    qv = bi.encode([QUERY_PREFIX + q], normalize_embeddings=True).astype("float32")
    assert list(index.search(qv, TOP_K)[1][0]) == list(np.argsort(-(emb @ qv[0]))[:TOP_K])

def retrieve(query):
    """Pipeline 1, first stage: top-K chunks by cosine similarity."""
    t0 = time.perf_counter()
    q = bi.encode([QUERY_PREFIX + query], normalize_embeddings=True).astype("float32")
    t1 = time.perf_counter()
    scores, ids = index.search(q, TOP_K)
    t2 = time.perf_counter()
    return list(ids[0]), list(scores[0]), {"embed_s": t1 - t0, "search_s": t2 - t1}

def rerank(query, ids):
    """Pipeline 2, second stage: the same chunks, ordered by the cross-encoder."""
    t0 = time.perf_counter()
    s = cross.predict([(query, chunks[i]["text"]) for i in ids], batch_size=len(ids),
                      show_progress_bar=False)
    t1 = time.perf_counter()
    order = np.argsort(-s)
    return [ids[o] for o in order], [float(s[o]) for o in order], {"rerank_s": t1 - t0}

## 6. Answer generation

Both pipelines use this prompt and this model. Passages are numbered, the model may only use
them, and it is told to say so when they do not answer the question. Answers are saved in
`results/answers_cache.json`, keyed by model and prompt text, so a run without the API key
reproduces the same answers.

In [ ]:
SYSTEM = ("Answer the question using only the numbered passages. Cite the passages you use, "
          "like [1]. If the passages do not answer the question, say that the document "
          "does not cover it.")

def build_prompt(query, ids):
    ctx = "\n\n".join(f"[{n}] {chunks[i]['text']}" for n, i in enumerate(ids, 1))
    return f"Context:\n{ctx}\n\nQuestion: {query}"

def load_cache():
    if CACHE_PATH.exists():
        return json.loads(CACHE_PATH.read_text())
    try:
        return json.loads(urllib.request.urlopen(CACHE_URL, timeout=30).read())
    except Exception:
        return {}

def generate(user):
    key = hashlib.sha256(f"{GEN_MODEL}\n{SYSTEM}\n{user}".encode()).hexdigest()
    cache = load_cache()
    if key in cache:
        return cache[key]["answer"], cache[key]["gen_s"], True
    api_key = os.environ.get("OPENCODE_API_KEY")
    if not api_key:
        raise RuntimeError("no saved answer for this prompt and OPENCODE_API_KEY is not set")
    body = json.dumps({"model": GEN_MODEL, "max_tokens": 800, "system": SYSTEM,
                       "messages": [{"role": "user", "content": user}]}).encode()
    req = urllib.request.Request(GEN_URL, data=body, method="POST", headers={
        "content-type": "application/json", "x-api-key": api_key, "anthropic-version": "2023-06-01",
        "user-agent": "rag-rerank-eval/1.0", "x-opencode-session": SESSION})
    t0 = time.perf_counter()
    reply = json.loads(urllib.request.urlopen(req, timeout=120).read())
    gen_s = time.perf_counter() - t0
    answer = "".join(b.get("text", "") for b in reply["content"] if b["type"] == "text").strip()
    cache[key] = {"model": GEN_MODEL, "date": date.today().isoformat(), "answer": answer,
                  "gen_s": gen_s, "usage": reply.get("usage")}
    CACHE_PATH.parent.mkdir(exist_ok=True)
    CACHE_PATH.write_text(json.dumps(cache, indent=1))
    return answer, gen_s, False

SESSION = str(uuid.uuid4())

## 7. Running a query through both pipelines

In [ ]:
def run_query(query, with_answers=True):
    ids, sims, t_ret = retrieve(query)
    new_ids, ce_scores, t_rr = rerank(query, ids)
    res = {"query": query, "baseline_ids": ids[:TOP_N], "rerank_ids": new_ids[:TOP_N],
           "rank_shift": {chunks[i]["id"]: (ids.index(i) + 1, new_ids.index(i) + 1) for i in ids},
           "cosine": dict(zip((chunks[i]["id"] for i in ids), sims)),
           "cross_score": dict(zip((chunks[i]["id"] for i in new_ids), ce_scores)),
           "time": {**t_ret, **t_rr}}
    if with_answers:
        res["answer_1"], res["gen_s_1"], _ = generate(build_prompt(query, res["baseline_ids"]))
        res["answer_2"], res["gen_s_2"], _ = generate(build_prompt(query, res["rerank_ids"]))
    return res

def show(res):
    print("QUERY:", res["query"])
    print(f"{'chunk':<14}{'baseline':>9}{'rerank':>8}{'cosine':>9}{'cross':>9}")
    for cid, (a, b) in sorted(res["rank_shift"].items(), key=lambda kv: kv[1][1]):
        print(f"{cid:<14}{a:>9}{b:>8}{res['cosine'][cid]:>9.3f}{res['cross_score'][cid]:>9.2f}")
    for k in ("answer_1", "answer_2"):
        if k in res:
            print(f"\n{k.upper().replace('_', ' ')}:\n{res[k]}")

## 8. Latency profile

Each stage is timed separately, after one warm-up call, with the models already loaded. The
numbers depend on the machine, so the hardware is printed with them.

In [ ]:
import platform

def profile(queries, repeats=5):
    retrieve(queries[0]); rerank(queries[0], retrieve(queries[0])[0])      # warm-up
    rows = {"embed_s": [], "search_s": [], "rerank_s": []}
    for q in queries:
        for _ in range(repeats):
            ids, _, t1 = retrieve(q)
            _, _, t2 = rerank(q, ids)
            for k, v in {**t1, **t2}.items():
                rows[k].append(v)
    ms = {k: statistics.median(v) * 1000 for k, v in rows.items()}
    print(f"{platform.machine()} | {platform.processor() or platform.platform()} | torch {torch.__version__}")
    print(f"{len(queries)} queries x {repeats} runs, median per stage")
    print(f"  baseline retrieval (embed question + search): {ms['embed_s'] + ms['search_s']:8.1f} ms")
    print(f"  re-ranking ({TOP_K} pairs):                    {ms['rerank_s']:8.1f} ms")
    print(f"  pipeline 2 total (retrieval + re-ranking):    {ms['embed_s'] + ms['search_s'] + ms['rerank_s']:8.1f} ms")
    return ms

## 9. Queries and results

Queries, rank shifts and the written analysis are added below.